# MeetMind — Entraînement et évaluation sur Google Colab (GPU)

**Projet NLP · Yousra Erraki · 2026-2027**

Ce notebook fait, dans l'ordre :
1. Installation du projet et des modèles (Whisper, BARThez, mT5, CamemBERT, spaCy, Sentence-BERT)
2. **Fine-tuning de CamemBERT** (classification des phrases : information / décision / action)
3. **Tableau comparatif des modèles de résumé** (TextRank vs BARThez vs mT5) — ROUGE-L, BERTScore, temps
4. **Évaluation du pipeline complet** avec les vrais modèles (objectifs : ROUGE-L ≥ 0,35 · F1 actions ≥ 0,75)
5. Test de **Whisper** (+ diarisation) sur un audio
6. Téléchargement du modèle entraîné → à copier dans `models/camembert-meetmind` du projet

> **Avant de commencer :** menu *Exécution → Modifier le type d'exécution → GPU (T4)*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Pas de GPU : activez-le dans Exécution → Modifier le type d'exécution"

## 1. Installation
Chargez `meetmind.zip` (le projet) quand la fenêtre d'upload apparaît.

In [ ]:
from google.colab import files
import os, zipfile
if not os.path.exists("meetmind"):
    up = files.upload()                      # choisir meetmind.zip
    zipfile.ZipFile(next(iter(up))).extractall(".")
%cd meetmind
!ls

In [ ]:
%%capture install
!pip install -q -r requirements.txt -r requirements-nlp.txt
!python -m spacy download fr_core_news_lg
!apt-get -qq install -y ffmpeg
import nltk; nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)

In [ ]:
print(install.stdout[-1500:] if install.stdout else "installation terminée")

## 2. Fine-tuning de CamemBERT

Données d'entraînement :
- `data/annotations/seed.jsonl` — exemples écrits à la main
- `data/annotations/reunions_exemple.csv` — phrases des réunions d'exemple, vérifiées
- **vos annotations** : lancez `training/prepare_data.py` sur vos transcriptions, corrigez la colonne `label`, puis ajoutez le fichier ci-dessous.

⚠️ Avec ~280 exemples, le modèle reste fragile : visez **300 à 500 phrases réelles** annotées.

In [ ]:
# (optionnel) pré-annoter vos propres transcriptions, puis les corriger à la main
# !python training/prepare_data.py data/transcripts/*.txt -o data/annotations/mes_reunions.csv

# Corpus synthétique (10 réunions annotées) + exemples écrits à la main
DATA = "data/corpus/annotations_train.csv data/annotations/seed.jsonl data/annotations/reunions_exemple.csv"
# DATA += " data/annotations/mes_reunions.csv"      # ← vos réunions réelles annotées
!python training/train_classifier.py --data $DATA --epochs 6 --batch 16 --lr 3e-5

# Comparaison avec le classifieur léger TF-IDF (même données, même test)
!python training/train_tfidf.py --data $DATA --eval data/corpus/annotations_test.csv

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model="models/camembert-meetmind")
tests = [
    "On a décidé de passer à PostgreSQL.",
    "Imane, tu peux envoyer le devis au client avant jeudi ?",
    "Le serveur a redémarré deux fois hier.",
    "Je m'occupe de la présentation pour lundi.",
    "Ttafa9na nbeddlo la date dyal la démo.",
]
for t in tests:
    p = clf(t)[0]; print(f"{p['label']:<12} {p['score']:.2f}  {t}")

## 3. Tableau comparatif des modèles de résumé
Compare le résumé extractif (TextRank) à BARThez et mT5 sur les réunions annotées (`*.gold.json`). Ajoutez `llm` si un serveur Ollama est disponible.

In [ ]:
!python evaluation/compare_summarizers.py data/corpus/test/*.gold.json --models extractive barthez mt5

In [ ]:
import json, pandas as pd
rows = json.load(open("evaluation/comparaison_resumes.json"))
pd.DataFrame([{k: v for k, v in r.items() if k != "details"} for r in rows])

In [ ]:
# Lire les résumés produits par chaque modèle pour une réunion
for r in rows:
    print(f"=== {r['modele']}\n{r['details'][0]['summary']}\n")

## 4. Évaluation du pipeline complet (vrais modèles)
Choisissez le meilleur modèle de résumé d'après le tableau ci-dessus.

In [ ]:
import os
os.environ["NLP_MODE"] = "auto"
os.environ["SUMMARIZER"] = "barthez"   # ou "mt5"
os.environ["SUMMARIZER_MODEL"] = "moussaKam/barthez-orangesum-abstract"   # mT5 : csebuetnlp/mT5_multilingual_XLSum
# Jeu de TEST séparé : 5 réunions jamais vues pendant le développement
!NLP_MODE=auto python evaluation/evaluate.py data/corpus/test/*.gold.json --sentences data/corpus/annotations_test.csv --report evaluation/resultats_modeles.json

> ⚠️ **Lecture honnête des scores :** les réunions de `data/samples/` ont servi à mettre au point les règles.
> Pour le rapport final, évaluez sur un **jeu de test séparé** : vos propres réunions annotées, ou des extraits
> du corpus AMI / QMSum (traduits ou en anglais avec `SUMMARIZER=bart`).

## 5. Whisper (+ diarisation) sur un audio
Pour la diarisation : créez un token sur huggingface.co, acceptez les conditions de `pyannote/speaker-diarization-3.1` et collez le token ci-dessous.

In [ ]:
os.environ["WHISPER_MODEL"] = "small"     # "medium" = plus précis, plus lent
os.environ["DIARIZATION"] = "off"         # "on" pour activer pyannote
os.environ["HF_TOKEN"] = ""               # requis si DIARIZATION=on
audio = files.upload()                    # un .mp3 / .wav de réunion
audio_path = next(iter(audio))

In [ ]:
import time
from datetime import date
from backend.app.nlp.pipeline import run_pipeline
t = time.time()
rep = run_pipeline(audio_path=audio_path, title="Test audio", meeting_date=date.today(), audio_language="fr")
print(f"Traitement : {time.time() - t:.1f} s   (objectif : < 60 s pour 1 h de réunion)")
print("Modèles :", rep.models_used, "\n")
print("RÉSUMÉ :", rep.summary, "\n")
print("DÉCISIONS :"); [print(" •", d.text) for d in rep.decisions]
print("ACTIONS :");  [print(f" • {a.task} — {a.owner or '?'} — {a.deadline_date or a.deadline or '—'}") for a in rep.actions]

## 6. Récupérer le modèle entraîné
Dézippez-le dans le dossier `models/` du projet sur votre PC : l'API le charge automatiquement (`CLASSIFIER_MODEL_PATH`).

In [ ]:
!cd models && zip -qr ../camembert-meetmind.zip camembert-meetmind
files.download("camembert-meetmind.zip")